# 02 Bereinigung

Ziel: Die Auffälligkeiten aus Notebook 01 beheben, die Tabellen verbinden und den Gewinn berechnen. Die Rohdaten in `data/raw` bleiben unverändert.

In [1]:
import pandas as pd

products = pd.read_csv("../data/raw/products.csv")
stores = pd.read_csv("../data/raw/stores.csv")
sales = pd.read_csv("../data/raw/sales.csv")
inventory = pd.read_csv("../data/raw/inventory.csv")
calendar = pd.read_csv("../data/raw/calendar.csv")

## 1. products: Preise in Zahlen umwandeln
`Product_Cost` und `Product_Price` enthalten ein Dollarzeichen und sind deshalb Text. Das `$` wird entfernt, danach werden die Spalten in Dezimalzahlen (`float`) umgewandelt.

In [2]:
products["Product_Cost"] = products["Product_Cost"].str.replace("$", "", regex=False)
products["Product_Cost"] = products["Product_Cost"].astype(float)

products["Product_Price"] = products["Product_Price"].str.replace("$", "", regex=False)
products["Product_Price"] = products["Product_Price"].astype(float)

In [3]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Product_ID        35 non-null     int64  
 1   Product_Name      35 non-null     str    
 2   Product_Category  35 non-null     str    
 3   Product_Cost      35 non-null     float64
 4   Product_Price     35 non-null     float64
dtypes: float64(2), int64(1), str(2)
memory usage: 1.5 KB


In [4]:
products.head()

,Product_ID,Product_Name,Product_Category,Product_Cost,Product_Price
0,1,Action Figure,Toys,9.99,15.99
1,2,Animal Figures,Toys,9.99,12.99
2,3,Barrel O' Slime,Art & Crafts,1.99,3.99
3,4,Chutes & Ladders,Games,9.99,12.99
4,5,Classic Dominoes,Games,7.99,9.99


## 2. Datumsspalten umwandeln
`Store_Open_Date` (stores), `Date` (sales) und `Date` (calendar) sind Text und müssen in echte Datumswerte umgewandelt werden. In `calendar` steht das Datum als Monat/Tag/Jahr, deshalb wird dort das Format angegeben.

In [5]:
stores["Store_Open_Date"] = pd.to_datetime(stores["Store_Open_Date"])
sales["Date"] = pd.to_datetime(sales["Date"])
calendar["Date"] = pd.to_datetime(calendar["Date"], format="%m/%d/%Y")

In [6]:
stores.info()
sales.info()
calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Store_ID         50 non-null     int64         
 1   Store_Name       50 non-null     str           
 2   Store_City       50 non-null     str           
 3   Store_Location   50 non-null     str           
 4   Store_Open_Date  50 non-null     datetime64[us]
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 2.1 KB
<class 'pandas.DataFrame'>
RangeIndex: 829262 entries, 0 to 829261
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   Sale_ID     829262 non-null  int64         
 1   Date        829262 non-null  datetime64[us]
 2   Store_ID    829262 non-null  int64         
 3   Product_ID  829262 non-null  int64         
 4   Units       829262 non-null  int64         
dtypes: datetime64[us](1), int6

## 3. Tabellen verbinden (SQL)
`sales` enthält nur IDs und Stückzahlen. Damit Preise, Kategorien und Filialinformationen zur Verfügung stehen, werden die Tabellen in eine SQLite-Datenbank (im Arbeitsspeicher) geschrieben und mit einem JOIN verbunden. Danach muss die Zeilenzahl unverändert 829.262 betragen.

In [7]:
from pathlib import Path
from sqlalchemy import create_engine, inspect

Path("../data/processed").mkdir(parents=True, exist_ok=True)

engine = create_engine("sqlite:///../data/processed/maven_toys.db")
connection = engine.connect()

products.to_sql("products", engine, index=False, if_exists="replace")
stores.to_sql("stores", engine, index=False, if_exists="replace")
sales.to_sql("sales", engine, index=False, if_exists="replace")
inventory.to_sql("inventory", engine, index=False, if_exists="replace")

print(inspect(engine).get_table_names())

['inventory', 'inventory_full', 'products', 'sales', 'sales_full', 'stores']


In [8]:
query = """
SELECT
    s.Sale_ID,
    s.Date,
    s.Units,
    s.Product_ID,
    p.Product_Name,
    p.Product_Category,
    p.Product_Cost,
    p.Product_Price,
    s.Store_ID,
    st.Store_Name,
    st.Store_City,
    st.Store_Location
FROM sales AS s
JOIN products AS p ON s.Product_ID = p.Product_ID
JOIN stores AS st ON s.Store_ID = st.Store_ID
"""

sales_full = pd.read_sql(query, connection, parse_dates=["Date"])

In [9]:
print("Zeilen sales:", len(sales))
print("Zeilen sales_full:", len(sales_full))
sales_full.head()

Zeilen sales: 829262
Zeilen sales_full: 829262


,Sale_ID,Date,Units,Product_ID,Product_Name,Product_Category,Product_Cost,Product_Price,Store_ID,Store_Name,Store_City,Store_Location
0,1,2022-01-01,1,4,Chutes & Ladders,Games,9.99,12.99,24,Maven Toys Aguascalientes 1,Aguascalientes,Downtown
1,2,2022-01-01,1,1,Action Figure,Toys,9.99,15.99,28,Maven Toys Puebla 2,Puebla,Downtown
2,3,2022-01-01,1,8,Deck Of Cards,Games,3.99,6.99,6,Maven Toys Mexicali 1,Mexicali,Commercial
3,4,2022-01-01,1,7,Dart Gun,Sports & Outdoors,11.99,15.99,48,Maven Toys Saltillo 2,Saltillo,Commercial
4,5,2022-01-01,1,18,Lego Bricks,Toys,34.99,39.99,44,Maven Toys Puebla 3,Puebla,Residential


## 4. Umsatz, Kosten und Gewinn berechnen
Pro Verkaufszeile werden Umsatz, Kosten und Gewinn berechnet:
- Umsatz = Stückzahl mal Verkaufspreis
- Kosten = Stückzahl mal Einkaufspreis
- Gewinn = Umsatz minus Kosten

In [10]:
sales_full["Revenue"] = sales_full["Units"] * sales_full["Product_Price"]
sales_full["Total_Cost"] = sales_full["Units"] * sales_full["Product_Cost"]
sales_full["Profit"] = sales_full["Revenue"] - sales_full["Total_Cost"]

### Prüfung `Units`
`Units` ist nicht immer 1. Bei rund 83 % der Verkäufe wurde ein Stück verkauft, bei rund 17 % zwei bis 30 Stück. Umsatz und Kosten müssen deshalb mit `Units` multipliziert werden.

In [11]:
sales_full["Units"].value_counts()

Units
1     689023
3      65638
2      54576
4      13117
6       2808
5       2628
7        779
9        203
8        180
10        96
11        36
12        35
13        25
15        18
17        13
21        12
16        12
19        11
18        10
20        10
14         9
24         7
22         7
23         5
30         2
27         1
25         1
Name: count, dtype: int64

In [12]:
umsatz = sales_full["Revenue"].sum()
gewinn = sales_full["Profit"].sum()

print("Umsatz:", round(umsatz, 2))
print("Gewinn:", round(gewinn, 2))
print("Marge in %:", round(gewinn / umsatz * 100, 1))

Umsatz: 14444572.35
Gewinn: 4014029.0
Marge in %: 27.8


In [13]:
sales_full.head()

,Sale_ID,Date,Units,Product_ID,Product_Name,Product_Category,Product_Cost,Product_Price,Store_ID,Store_Name,Store_City,Store_Location,Revenue,Total_Cost,Profit
0,1,2022-01-01,1,4,Chutes & Ladders,Games,9.99,12.99,24,Maven Toys Aguascalientes 1,Aguascalientes,Downtown,12.99,9.99,3.0
1,2,2022-01-01,1,1,Action Figure,Toys,9.99,15.99,28,Maven Toys Puebla 2,Puebla,Downtown,15.99,9.99,6.0
2,3,2022-01-01,1,8,Deck Of Cards,Games,3.99,6.99,6,Maven Toys Mexicali 1,Mexicali,Commercial,6.99,3.99,3.0
3,4,2022-01-01,1,7,Dart Gun,Sports & Outdoors,11.99,15.99,48,Maven Toys Saltillo 2,Saltillo,Commercial,15.99,11.99,4.0
4,5,2022-01-01,1,18,Lego Bricks,Toys,34.99,39.99,44,Maven Toys Puebla 3,Puebla,Residential,39.99,34.99,5.0


## 5. Ergebnis speichern
Die verbundene Tabelle mit Umsatz, Kosten und Gewinn wird als Tabelle `sales_full` in der Datenbank gespeichert. Sie ist die Grundlage für die Auswertungen in den folgenden Notebooks.

In [14]:
sales_full.to_sql("sales_full", engine, index=False, if_exists="replace")

print(inspect(engine).get_table_names())

['inventory', 'inventory_full', 'products', 'sales', 'sales_full', 'stores']


## 6. Lagerbestand verbinden und Lagerwert berechnen
`inventory` enthält nur IDs und den Bestand (`Stock_On_Hand`). Die Tabelle wird per JOIN mit `products` und `stores` verbunden. Der Lagerwert (Kapitalbindung) ergibt sich aus Bestand mal Einkaufspreis. Die Daten sind eine Momentaufnahme ohne Datum.x

In [15]:
query_inv = """
SELECT
    i.Store_ID,
    st.Store_Name,
    st.Store_City,
    st.Store_Location,
    i.Product_ID,
    p.Product_Name,
    p.Product_Category,
    p.Product_Cost,
    p.Product_Price,
    i.Stock_On_Hand
FROM inventory AS i
JOIN products AS p ON i.Product_ID = p.Product_ID
JOIN stores AS st ON i.Store_ID = st.Store_ID
"""

inventory_full = pd.read_sql(query_inv, connection)

In [16]:
inventory_full["Stock_Value"] = inventory_full["Stock_On_Hand"] * inventory_full["Product_Cost"]

print("Zeilen inventory:", len(inventory))
print("Zeilen inventory_full:", len(inventory_full))
print("Gesamter Lagerwert:", round(inventory_full["Stock_Value"].sum(), 2))

Zeilen inventory: 1593
Zeilen inventory_full: 1593
Gesamter Lagerwert: 300209.58


In [17]:
inventory_full.to_sql("inventory_full", engine, index=False, if_exists="replace")

print(inspect(engine).get_table_names())

['inventory', 'inventory_full', 'products', 'sales', 'sales_full', 'stores']


## Zusammenfassung

**Ergebnis der Bereinigung:**
- Preise in `products` in Zahlen umgewandelt, Datumsspalten in `stores`, `sales` und `calendar` in Datumswerte.
- Alle Tabellen in der SQLite-Datenbank `data/processed/maven_toys.db` gespeichert (nicht im Repository).
- `sales` per SQL-JOIN mit `products` und `stores` zu `sales_full` verbunden: 829.262 Zeilen, keine Zeile verloren oder verdoppelt.
- `Units` ist nicht immer 1 (rund 83 % gleich 1, Rest 2 bis 30), daher Umsatz und Kosten mit `Units` multipliziert.
- `inventory` per SQL-JOIN zu `inventory_full` verbunden: 1.593 Zeilen, Lagerwert (Bestand mal Einkaufspreis) als `Stock_Value`.

**Kennzahlen gesamt (01.01.2022 bis 30.09.2023):**
- Umsatz: rund 14,44 Mio. USD
- Gewinn: rund 4,01 Mio. USD
- Marge: 27,8 %
- Lagerwert (Stichtag): rund 300.210 USD

**Aufgabenliste für Notebook 03 (Auswertung):**
1. Daten aus der Datenbank laden (`sales_full`, `inventory_full`), `Date` beim Einlesen in ein Datum umwandeln.
2. Teilfrage 1: Umsatz, Gewinn und Marge pro Monat berechnen und als Linie darstellen.
3. Saisonalität: Monate vergleichen. Achtung: 2023 hat nur Januar bis September, ein Jahresvergleich ist nur für diese neun Monate fair.
4. Teilfrage 2: Gewinn und Marge nach Produktkategorie, Produkt, Filialtyp (`Store_Location`) und Stadt.
5. Die wichtigsten Diagramme speichern und Auffälligkeiten festhalten.